In [88]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import soundfile as sf

plt.style.use("ggplot")
pd.set_option("display.max_columns", None)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [89]:
DATASET_PATH = r"D:\ML-PROJECTS\Emotion analysis\dataset\voice-Mozilla"

PROCESSED_AUDIO = os.path.join(DATASET_PATH, "processed_audio")

GENDER_OUTPUT = os.path.join(PROCESSED_AUDIO, "gender")

AGE_OUTPUT = os.path.join(PROCESSED_AUDIO, "age")

os.makedirs(GENDER_OUTPUT, exist_ok=True)
os.makedirs(AGE_OUTPUT, exist_ok=True)

print("Folders Created Successfully")

Folders Created Successfully


In [90]:
gender_df = pd.read_csv(
    os.path.join(DATASET_PATH, "processed", "gender_metadata.csv")
)

age_df = pd.read_csv(
    os.path.join(DATASET_PATH, "processed", "age_metadata.csv")
)

print("Gender Dataset :", gender_df.shape)
print("Age Dataset :", age_df.shape)

Gender Dataset : (73278, 8)
Age Dataset : (54593, 8)


In [91]:
def get_audio_path(filename):

    folder = filename.split("/")[0]

    return os.path.join(
        DATASET_PATH,
        folder,
        folder,
        os.path.basename(filename)
    )

In [92]:
sample = gender_df.iloc[0]["filename"]

audio_path = get_audio_path(sample)

print(audio_path)

print(os.path.exists(audio_path))

D:\ML-PROJECTS\Emotion analysis\dataset\voice-Mozilla\cv-valid-train\cv-valid-train\sample-000005.mp3
True


In [93]:
TARGET_SR = 16000


def preprocess_audio(audio_path):

    signal, sr = librosa.load(
        audio_path,
        sr=TARGET_SR,
        mono=True
    )

    signal, _ = librosa.effects.trim(
        signal,
        top_db=25
    )

    max_amp = np.max(np.abs(signal))

    if max_amp > 0:
        signal = signal / max_amp

    return signal

In [94]:
import soundfile as sf

def save_audio(signal, output_path):
    try:
        sf.write(
            output_path,
            signal,
            TARGET_SR
        )
        return True

    except Exception as e:
        print(f"Error saving {output_path}")
        print(e)
        return False
    

In [95]:
processed_gender = []

for _, row in tqdm(gender_df.iterrows(),
                   total=len(gender_df)):

    input_file = get_audio_path(row["filename"])

    output_name = (
        os.path.splitext(
            os.path.basename(input_file)
        )[0]
        + ".wav"
    )

    output_file = os.path.join(
        GENDER_OUTPUT,
        output_name
    )

    if os.path.exists(output_file):

        processed_gender.append(output_file)

        continue

    try:

        signal = preprocess_audio(input_file)

        save_audio(signal, output_file)

        processed_gender.append(output_file)

    except Exception as e:
        print("ERROR:", input_file)
        print(e)
        processed_gender.append(None)

100%|██████████| 73278/73278 [00:02<00:00, 31845.12it/s]


In [96]:
processed_age = []

for _, row in tqdm(age_df.iterrows(),
                   total=len(age_df)):

    input_file = get_audio_path(row["filename"])

    output_name = (
        os.path.splitext(
            os.path.basename(input_file)
        )[0]
        + ".wav"
    )

    output_file = os.path.join(
        AGE_OUTPUT,
        output_name
    )

    if os.path.exists(output_file):

        processed_age.append(output_file)

        continue

    try:

        signal = preprocess_audio(input_file)

        save_audio(signal, output_file)

        processed_age.append(output_file)

    except Exception as e:
        print("ERROR:", input_file)
        print(e)
        processed_gender.append(None)

100%|██████████| 54593/54593 [00:01<00:00, 35355.49it/s]


In [97]:
gender_df["processed_audio"] = processed_gender

age_df["processed_audio"] = processed_age

gender_df = gender_df[
    gender_df["processed_audio"].notna()
].copy()

age_df = age_df[
    age_df["processed_audio"].notna()
].copy()

In [98]:
gender_df.to_csv(
    os.path.join(
        DATASET_PATH,
        "processed",
        "gender_processed.csv"
    ),
    index=False
)

age_df.to_csv(
    os.path.join(
        DATASET_PATH,
        "processed",
        "age_processed.csv"
    ),
    index=False
)

print("Processed Metadata Saved Successfully")

Processed Metadata Saved Successfully


In [104]:
print(gender_df.head())

print()

print(age_df.head())

                           filename  \
0  cv-valid-train/sample-000005.mp3   
1  cv-valid-train/sample-000008.mp3   
2  cv-valid-train/sample-000013.mp3   
3  cv-valid-train/sample-000014.mp3   
4  cv-valid-train/sample-000019.mp3   

                                                text  up_votes  down_votes  \
0  a shepherd may like to travel but he should ne...         1           0   
1                      put jackie right on the staff         3           0   
2  but he had found a guide and didn't want to mi...         1           0   
3  as they began to decorate the hallway a silhou...         1           0   
4   then they got ahold of some dough and went goofy         1           0   

         age  gender     accent  duration  \
0   twenties  female         us       NaN   
1  seventies    male         us       NaN   
2   thirties  female         us       NaN   
3    sixties    male    england       NaN   
4    fifties    male  australia       NaN   

                         

In [105]:
print("Gender Dataset Shape:", gender_df.shape)
print("Age Dataset Shape:", age_df.shape)

Gender Dataset Shape: (73278, 9)
Age Dataset Shape: (54593, 9)


In [106]:
print(gender_df["processed_audio"].isnull().sum())
print(age_df["processed_audio"].isnull().sum())

0
0


In [107]:
import os

print(os.path.exists(gender_df.iloc[0]["processed_audio"]))
print(os.path.exists(age_df.iloc[0]["processed_audio"]))

True
True
